# Tutorial 03: Creating Custom PDPTW Problems

Learn how to create custom PDPTW (Pickup and Delivery Problem with Time Windows) instances from scratch for your specific use case.

**What you'll learn:**
- Build PDPTW instances manually by creating order tables
- Define distance and time matrices
- Load problem data from CSV files
- Modify existing instances
- Validate and troubleshoot problem definitions

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Basic understanding of pandas DataFrames

**Time:** ~25 minutes

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import random

# VRP Toolkit
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import ALNSSolver, greedy_insertion_initial_solution

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Imports successful! Ready to create custom problems.")

## 2. Quick Start: Simplest Custom Problem

Let's create the **simplest possible PDPTW instance**: 1 depot, 1 pickup-delivery pair.

A PDPTW instance needs three components:
1. **Order table** - Defines nodes (depot, pickups, deliveries)
2. **Distance matrix** - Distances between all nodes
3. **Time matrix** - Travel times between all nodes

In [ ]:
# Step 1: Create order table
order_table = pd.DataFrame([
    # Depot (ID=0)
    {'ID': 0, 'Type': 'depot', 'X': 0.0, 'Y': 0.0, 'Demand': 0.0,
     'StartTime': 0.0, 'EndTime': 480.0, 'ServiceTime': 0.0, 'PartnerID': 0,
     'RealIndex': 0, 'RealType': 'depot'},
    
    # Pickup (ID=1)
    {'ID': 1, 'Type': 'cp', 'X': 1.0, 'Y': 1.0, 'Demand': 10.0,
     'StartTime': 0.0, 'EndTime': 480.0, 'ServiceTime': 5.0, 'PartnerID': 2,
     'RealIndex': 1, 'RealType': 'cp'},
    
    # Delivery (ID=2)
    {'ID': 2, 'Type': 'cd', 'X': 2.0, 'Y': 2.0, 'Demand': -10.0,
     'StartTime': 0.0, 'EndTime': 480.0, 'ServiceTime': 5.0, 'PartnerID': 1,
     'RealIndex': 2, 'RealType': 'cd'}
])

# Step 2: Create distance matrix (Euclidean distances)
n_nodes = len(order_table)
distance_matrix = np.zeros((n_nodes, n_nodes))

for i in range(n_nodes):
    for j in range(n_nodes):
        if i != j:
            x_diff = order_table.iloc[i]['X'] - order_table.iloc[j]['X']
            y_diff = order_table.iloc[i]['Y'] - order_table.iloc[j]['Y']
            distance_matrix[i, j] = np.sqrt(x_diff**2 + y_diff**2)

# Step 3: Create time matrix (distance / speed)
robot_speed = 1.0  # 1 distance unit per minute
time_matrix = distance_matrix / robot_speed

# Step 4: Create PDPTWInstance
instance = PDPTWInstance(
    order_table=order_table,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=robot_speed
)

print(f"Custom PDPTW instance created!")
print(f"  - Nodes: {len(instance.indices)}")
print(f"  - Orders: {instance.n}")
print(f"\nOrder table:")
print(order_table[['ID', 'Type', 'X', 'Y', 'Demand', 'PartnerID']])

**What just happened:**
1. We created an order table with 3 nodes: depot (ID=0), pickup (ID=1), delivery (ID=2)
2. Pickup and delivery are linked via `PartnerID`
3. We computed distances using Euclidean formula
4. We created a `PDPTWInstance` that can be solved with ALNS

**Key insight:** `RealIndex` maps each node ID to its position in the distance/time matrices. For simple cases, `RealIndex = ID`.

## 3. Understanding the Order Table Structure

The order table is the heart of a PDPTW instance. Let's understand each column:

### 3.1 Required Columns

In [ ]:
# Column descriptions
column_info = pd.DataFrame([
    {'Column': 'ID', 'Type': 'int', 'Description': 'Sequential node ID (0, 1, 2, ...)'},
    {'Column': 'Type', 'Type': 'str', 'Description': "'depot', 'cp' (pickup), 'cd' (delivery), 'charging'"},
    {'Column': 'X, Y', 'Type': 'float', 'Description': 'Node coordinates (for visualization)'},
    {'Column': 'Demand', 'Type': 'float', 'Description': 'Positive for pickup, negative for delivery, 0 for depot'},
    {'Column': 'StartTime', 'Type': 'float', 'Description': 'Time window start (minutes)'},
    {'Column': 'EndTime', 'Type': 'float', 'Description': 'Time window end (minutes)'},
    {'Column': 'ServiceTime', 'Type': 'float', 'Description': 'Service duration at node (minutes)'},
    {'Column': 'PartnerID', 'Type': 'int', 'Description': 'ID of matching pickup/delivery node'},
    {'Column': 'RealIndex', 'Type': 'int', 'Description': 'Index in distance/time matrix'},
    {'Column': 'RealType', 'Type': 'str', 'Description': 'Original node type (for reference)'}
])

print("PDPTW Order Table Columns:")
print(column_info.to_string(index=False))

### 3.2 Node Type Rules

**Critical constraints:**
1. **Depot** (Type='depot'): Must have ID=0, Demand=0, PartnerID=0
2. **Pickup** (Type='cp'): Demand > 0, PartnerID points to corresponding delivery
3. **Delivery** (Type='cd'): Demand < 0 (negative of pickup), PartnerID points back to pickup
4. **Charging** (Type='charging', optional): Demand=0, used for battery recharging

**ID numbering convention:**
- Depot: ID=0
- Pickups: ID=1 to n
- Deliveries: ID=(n+1) to 2n
- Charging (optional): ID=2n+1

In [ ]:
# Example: 2 orders with correct ID numbering
n_orders = 2

order_data = [
    # Depot
    {'ID': 0, 'Type': 'depot', 'Demand': 0.0, 'PartnerID': 0},
    
    # Pickups (ID: 1 to n)
    {'ID': 1, 'Type': 'cp', 'Demand': 10.0, 'PartnerID': 3},  # Links to delivery ID=3
    {'ID': 2, 'Type': 'cp', 'Demand': 15.0, 'PartnerID': 4},  # Links to delivery ID=4
    
    # Deliveries (ID: n+1 to 2n)
    {'ID': 3, 'Type': 'cd', 'Demand': -10.0, 'PartnerID': 1},  # Links back to pickup ID=1
    {'ID': 4, 'Type': 'cd', 'Demand': -15.0, 'PartnerID': 2},  # Links back to pickup ID=2
]

demo_table = pd.DataFrame(order_data)
print("Correct ID numbering and PartnerID links:")
print(demo_table[['ID', 'Type', 'Demand', 'PartnerID']])

# Verify pickup-delivery pairing
print("\nVerification:")
for _, pickup in demo_table[demo_table['Type'] == 'cp'].iterrows():
    delivery_id = pickup['PartnerID']
    delivery = demo_table[demo_table['ID'] == delivery_id].iloc[0]
    print(f"Pickup {pickup['ID']} (demand={pickup['Demand']}) → Delivery {delivery_id} (demand={delivery['Demand']})")

## 4. Creating Multi-Order Instances

Let's create a realistic instance with multiple orders using a helper function:

### 4.1 Helper Function for Order Table Generation

In [ ]:
def create_order_table(depot_coords, pickup_coords, delivery_coords, 
                       demands=None, time_window=(0, 480), service_time=5.0):
    """
    Create order table from coordinate lists.
    
    Args:
        depot_coords: (x, y) tuple for depot
        pickup_coords: List of (x, y) tuples for pickups
        delivery_coords: List of (x, y) tuples for deliveries (must match pickups)
        demands: List of demand values (default: 10.0 for all)
        time_window: (start, end) time window for all nodes
        service_time: Service time for pickups/deliveries
    
    Returns:
        pd.DataFrame: Order table
    """
    if len(pickup_coords) != len(delivery_coords):
        raise ValueError("Number of pickups must equal number of deliveries")
    
    n_orders = len(pickup_coords)
    if demands is None:
        demands = [10.0] * n_orders
    
    rows = []
    
    # Depot
    rows.append({
        'ID': 0, 'Type': 'depot', 'X': depot_coords[0], 'Y': depot_coords[1],
        'Demand': 0.0, 'StartTime': time_window[0], 'EndTime': time_window[1],
        'ServiceTime': 0.0, 'PartnerID': 0, 'RealIndex': 0, 'RealType': 'depot'
    })
    
    # Pickups
    for i, (coords, demand) in enumerate(zip(pickup_coords, demands), 1):
        rows.append({
            'ID': i, 'Type': 'cp', 'X': coords[0], 'Y': coords[1],
            'Demand': demand, 'StartTime': time_window[0], 'EndTime': time_window[1],
            'ServiceTime': service_time, 'PartnerID': i + n_orders,
            'RealIndex': i, 'RealType': 'cp'
        })
    
    # Deliveries
    for i, (coords, demand) in enumerate(zip(delivery_coords, demands), 1):
        rows.append({
            'ID': i + n_orders, 'Type': 'cd', 'X': coords[0], 'Y': coords[1],
            'Demand': -demand, 'StartTime': time_window[0], 'EndTime': time_window[1],
            'ServiceTime': service_time, 'PartnerID': i,
            'RealIndex': i + n_orders, 'RealType': 'cd'
        })
    
    return pd.DataFrame(rows)

# Test the helper
test_table = create_order_table(
    depot_coords=(0, 0),
    pickup_coords=[(1, 2), (3, 1), (2, 4)],
    delivery_coords=[(4, 3), (5, 5), (1, 5)],
    demands=[10, 15, 20]
)

print("Generated order table for 3 orders:")
print(test_table[['ID', 'Type', 'X', 'Y', 'Demand', 'PartnerID']])

### 4.2 Creating Distance and Time Matrices

In [ ]:
def compute_euclidean_distance_matrix(order_table):
    """
    Compute Euclidean distance matrix from order table coordinates.
    
    Args:
        order_table: DataFrame with X, Y columns
    
    Returns:
        np.ndarray: Distance matrix
    """
    n = len(order_table)
    dist_matrix = np.zeros((n, n))
    
    coords = order_table[['X', 'Y']].values
    
    for i in range(n):
        for j in range(n):
            if i != j:
                dist_matrix[i, j] = np.linalg.norm(coords[i] - coords[j])
    
    return dist_matrix

def compute_time_matrix_from_distance(distance_matrix, speed):
    """
    Convert distance matrix to time matrix.
    
    Args:
        distance_matrix: Distance matrix
        speed: Travel speed (distance units per minute)
    
    Returns:
        np.ndarray: Time matrix (minutes)
    """
    return distance_matrix / speed

# Create complete instance
order_table = create_order_table(
    depot_coords=(0, 0),
    pickup_coords=[(2, 3), (5, 1), (1, 5)],
    delivery_coords=[(6, 4), (3, 6), (4, 2)],
    demands=[12, 8, 15]
)

distance_matrix = compute_euclidean_distance_matrix(order_table)
time_matrix = compute_time_matrix_from_distance(distance_matrix, speed=1.5)

instance = PDPTWInstance(
    order_table=order_table,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=1.5
)

print(f"Created instance with {instance.n} orders")
print(f"\nDistance matrix (first 4x4):")
print(distance_matrix[:4, :4].round(2))
print(f"\nTime matrix (first 4x4):")
print(time_matrix[:4, :4].round(2))

## 5. Loading from CSV Files

For larger instances, it's convenient to load data from CSV files.

### 5.1 CSV Format

In [ ]:
# Create example CSV file
import io

csv_content = """ID,Type,X,Y,Demand,StartTime,EndTime,ServiceTime,PartnerID,RealIndex,RealType
0,depot,0.0,0.0,0.0,0.0,480.0,0.0,0,0,depot
1,cp,2.5,3.5,10.0,0.0,480.0,5.0,3,1,cp
2,cp,5.0,1.5,15.0,0.0,480.0,5.0,4,2,cp
3,cd,6.0,4.5,-10.0,0.0,480.0,5.0,1,3,cd
4,cd,3.5,6.0,-15.0,0.0,480.0,5.0,2,4,cd
"""

# Load from CSV (in practice, use pd.read_csv('filename.csv'))
order_table_csv = pd.read_csv(io.StringIO(csv_content))

print("Loaded order table from CSV:")
print(order_table_csv)

# Create distance and time matrices
distance_matrix_csv = compute_euclidean_distance_matrix(order_table_csv)
time_matrix_csv = compute_time_matrix_from_distance(distance_matrix_csv, speed=2.0)

# Create instance
instance_csv = PDPTWInstance(
    order_table=order_table_csv,
    distance_matrix=distance_matrix_csv,
    time_matrix=time_matrix_csv,
    robot_speed=2.0
)

print(f"\nInstance created: {instance_csv.n} orders, {len(instance_csv.indices)} nodes")

### 5.2 Saving Instances to CSV

In [ ]:
# Save order table
# order_table.to_csv('my_instance.csv', index=False)

# For complete instances, save matrices too
# np.save('distance_matrix.npy', distance_matrix)
# np.save('time_matrix.npy', time_matrix)

# Loading back:
# order_table_loaded = pd.read_csv('my_instance.csv')
# distance_matrix_loaded = np.load('distance_matrix.npy')
# time_matrix_loaded = np.load('time_matrix.npy')

print("To save an instance:")
print("  order_table.to_csv('instance.csv', index=False)")
print("  np.save('distance.npy', distance_matrix)")
print("  np.save('time.npy', time_matrix)")

## 6. Modifying Existing Instances

Sometimes you want to modify an existing instance (e.g., adjust time windows, change demands).

### 6.1 Adjusting Time Windows

In [ ]:
# Start with existing instance
modified_table = order_table.copy()

# Make time windows tighter for pickups
pickup_mask = modified_table['Type'] == 'cp'
modified_table.loc[pickup_mask, 'StartTime'] = 60.0
modified_table.loc[pickup_mask, 'EndTime'] = 180.0

print("Modified time windows for pickups:")
print(modified_table[pickup_mask][['ID', 'Type', 'StartTime', 'EndTime']])

# Create new instance
instance_modified = PDPTWInstance(
    order_table=modified_table,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=1.5
)

print(f"\nModified instance created with tighter time windows")

### 6.2 Changing Demands

In [ ]:
# Double all demands
modified_demands = order_table.copy()

# Pickups: multiply positive demands
pickup_mask = modified_demands['Type'] == 'cp'
modified_demands.loc[pickup_mask, 'Demand'] *= 2

# Deliveries: multiply negative demands
delivery_mask = modified_demands['Type'] == 'cd'
modified_demands.loc[delivery_mask, 'Demand'] *= 2

print("Demand changes:")
print("Before:", order_table[order_table['Type'] == 'cp']['Demand'].values)
print("After:", modified_demands[modified_demands['Type'] == 'cp']['Demand'].values)

# Create instance with modified demands
instance_demands = PDPTWInstance(
    order_table=modified_demands,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=1.5
)

print(f"\nInstance created with doubled demands")

### 6.3 Adding Charging Station

In [ ]:
# Start with existing table
with_charging = order_table.copy()

# Add charging station node
charging_id = len(with_charging)
charging_row = pd.DataFrame([{
    'ID': charging_id,
    'Type': 'charging',
    'X': 3.0,
    'Y': 3.0,
    'Demand': 0.0,
    'StartTime': 0.0,
    'EndTime': 480.0,
    'ServiceTime': 0.0,
    'PartnerID': 0,
    'RealIndex': charging_id,
    'RealType': 'charging'
}])

with_charging = pd.concat([with_charging, charging_row], ignore_index=True)

# Recompute matrices to include charging station
distance_with_charging = compute_euclidean_distance_matrix(with_charging)
time_with_charging = compute_time_matrix_from_distance(distance_with_charging, speed=1.5)

instance_charging = PDPTWInstance(
    order_table=with_charging,
    distance_matrix=distance_with_charging,
    time_matrix=time_with_charging,
    robot_speed=1.5
)

print(f"Instance with charging station:")
print(with_charging[['ID', 'Type', 'X', 'Y']])

## 7. Validation and Troubleshooting

### 7.1 Common Issues and How to Fix Them

In [ ]:
def validate_pdptw_order_table(order_table):
    """
    Validate PDPTW order table for common issues.
    
    Returns:
        bool: True if valid, False otherwise (with error messages)
    """
    errors = []
    
    # Check 1: Depot exists and has ID=0
    if 0 not in order_table['ID'].values:
        errors.append("Missing depot (ID=0)")
    elif order_table[order_table['ID'] == 0]['Type'].values[0] != 'depot':
        errors.append("Node ID=0 must be depot")
    
    # Check 2: All pickups have matching deliveries
    pickups = order_table[order_table['Type'] == 'cp']
    deliveries = order_table[order_table['Type'] == 'cd']
    
    if len(pickups) != len(deliveries):
        errors.append(f"Pickup count ({len(pickups)}) != Delivery count ({len(deliveries)})")
    
    # Check 3: PartnerID links are correct
    for _, pickup in pickups.iterrows():
        partner_id = pickup['PartnerID']
        if partner_id not in order_table['ID'].values:
            errors.append(f"Pickup {pickup['ID']} has invalid PartnerID {partner_id}")
        else:
            partner = order_table[order_table['ID'] == partner_id].iloc[0]
            if partner['Type'] != 'cd':
                errors.append(f"Pickup {pickup['ID']} PartnerID points to non-delivery")
            if partner['PartnerID'] != pickup['ID']:
                errors.append(f"Pickup {pickup['ID']} and Delivery {partner_id} not linked")
    
    # Check 4: Demands match
    for _, pickup in pickups.iterrows():
        partner_id = pickup['PartnerID']
        if partner_id in order_table['ID'].values:
            delivery = order_table[order_table['ID'] == partner_id].iloc[0]
            if abs(pickup['Demand'] + delivery['Demand']) > 0.01:
                errors.append(f"Pickup {pickup['ID']} demand doesn't match delivery {partner_id}")
    
    # Check 5: Required columns exist
    required_cols = ['ID', 'Type', 'X', 'Y', 'Demand', 'StartTime', 'EndTime', 
                     'ServiceTime', 'PartnerID', 'RealIndex', 'RealType']
    missing_cols = set(required_cols) - set(order_table.columns)
    if missing_cols:
        errors.append(f"Missing columns: {missing_cols}")
    
    # Print results
    if errors:
        print("❌ Validation FAILED:")
        for error in errors:
            print(f"  - {error}")
        return False
    else:
        print("✅ Validation PASSED: Order table is valid")
        return True

# Test validation
print("Validating correct order table:")
validate_pdptw_order_table(order_table)

print("\nValidating intentionally broken table:")
broken_table = order_table.copy()
broken_table.loc[broken_table['ID'] == 1, 'PartnerID'] = 999  # Invalid partner
validate_pdptw_order_table(broken_table)

## 8. Solving Your Custom Instance

Once you've created a valid instance, you can solve it with ALNS:

In [ ]:
# Use our custom 3-order instance
print(f"Solving custom instance: {instance.n} orders, {len(instance.indices)} nodes")

# Problem parameters
num_vehicles = 2
vehicle_capacity = 30
battery_capacity = 200
battery_consume_rate = 1
penalty_unvisit = 1000
penalty_delay = 50

# Generate initial solution
initial_solution = greedy_insertion_initial_solution(
    problem=instance,
    num_vehicles=num_vehicles,
    vehicle_capacity=vehicle_capacity,
    battery_capacity=battery_capacity,
    battery_consume_rate=battery_consume_rate,
    penalty_unvisit=penalty_unvisit,
    penalty_delay=penalty_delay
)

print(f"\nInitial solution:")
print(f"  Objective: {initial_solution.objective_function():.2f}")
print(f"  Feasible: {initial_solution.is_feasible()}")
print(f"  Routes: {initial_solution.routes}")

## 9. Comparison and Best Practices

**When to create custom instances manually:**
- You have specific problem requirements
- You're modeling a real-world scenario
- You want full control over problem characteristics
- You're loading data from external sources (CSV, database)

**When to use data generators (Tutorial 07):**
- You need synthetic test data
- You want to generate problems at scale
- You're experimenting with different problem sizes
- You need randomized instances for benchmarking

**When to use OSMnx integration (Tutorial 02):**
- You're working with real geographic locations
- You need actual street network distances
- You're planning real delivery routes

**Common pitfalls and solutions:**

1. **Mismatched pickup-delivery demands**
   - Solution: Always ensure delivery demand = -pickup demand
   - Use validation function to catch this

2. **Incorrect PartnerID links**
   - Solution: Follow ID numbering convention (pickups 1 to n, deliveries n+1 to 2n)
   - Pickup i should link to delivery i+n

3. **Distance/time matrix size mismatch**
   - Solution: Matrix dimensions must equal number of nodes in order table
   - Recompute matrices if you add/remove nodes

4. **RealIndex confusion**
   - Solution: For simple cases, RealIndex = ID
   - Only differs when using external node IDs (like OSMnx)

## 10. Summary

**What you learned:**
- ✅ Create PDPTW instances from scratch using order tables
- ✅ Understand order table structure and column requirements
- ✅ Build distance and time matrices from coordinates
- ✅ Load and save instances from/to CSV files
- ✅ Modify existing instances (time windows, demands, charging stations)
- ✅ Validate instances to catch common errors

**Key takeaways:**
1. **Order table is the core** - Get the structure right (ID numbering, PartnerID links)
2. **Demand balance** - Pickup demand must equal delivery demand (opposite sign)
3. **Matrix consistency** - Distance/time matrix size must match number of nodes
4. **Validation is crucial** - Use validation function before solving

**Next steps:**
- Try Tutorial 04 for problem variants (CVRP, VRPTW)
- Try Tutorial 07 for automated data generation
- Experiment with your own problem scenarios
- Build a custom instance loader for your data format